# AI Loan Advisory Chatbot using Machine Learning and NLP

## Project Type
Ready-to-submit Colab project notebook

## Topic
AI Loan Advisory Chatbot using Machine Learning and NLP

## Objective
The objective of this project is to develop an AI-based loan advisory chatbot that can predict loan eligibility and provide basic loan-related guidance to users based on their financial details such as income, credit score, loan amount, employment status, and debt-to-income ratio.


## Methodology

1. Import required libraries  
2. Create/load loan applicant dataset  
3. Perform data preprocessing  
4. Conduct exploratory data analysis  
5. Apply feature engineering and scaling  
6. Train machine learning models  
7. Evaluate model performance  
8. Select best model  
9. Build rule-based NLP chatbot logic  
10. Test chatbot with sample user inputs  


In [ ]:
# ===============================
# 1. Import Required Libraries
# ===============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

## 2. Dataset Creation

For this project, a sample loan applicant dataset is created. It contains important financial and personal details required for loan eligibility prediction.


In [ ]:
# ===============================
# 2. Create Sample Dataset
# ===============================

np.random.seed(42)

n = 500

data = pd.DataFrame({
    'Age': np.random.randint(21, 60, n),
    'Gender': np.random.choice(['Male', 'Female'], n),
    'Married': np.random.choice(['Yes', 'No'], n),
    'Education': np.random.choice(['Graduate', 'Not Graduate'], n),
    'Employment_Status': np.random.choice(['Salaried', 'Self Employed', 'Unemployed'], n, p=[0.55, 0.35, 0.10]),
    'Applicant_Income': np.random.randint(15000, 150000, n),
    'Loan_Amount': np.random.randint(50000, 1000000, n),
    'Loan_Term_Months': np.random.choice([60, 120, 180, 240, 300, 360], n),
    'Credit_Score': np.random.randint(300, 900, n),
    'Existing_EMI': np.random.randint(0, 50000, n),
    'Property_Area': np.random.choice(['Urban', 'Semiurban', 'Rural'], n)
})

# Derived feature
data['Debt_to_Income_Ratio'] = data['Existing_EMI'] / data['Applicant_Income']

# Rule-based target creation for realistic loan approval behavior
def loan_status(row):
    if row['Credit_Score'] >= 700 and row['Applicant_Income'] >= 30000 and row['Debt_to_Income_Ratio'] < 0.45 and row['Employment_Status'] != 'Unemployed':
        return 1
    elif row['Credit_Score'] >= 650 and row['Applicant_Income'] >= 50000 and row['Debt_to_Income_Ratio'] < 0.55:
        return 1
    else:
        return 0

data['Loan_Status'] = data.apply(loan_status, axis=1)

data.head()

In [ ]:
# Check dataset shape and information
print("Dataset Shape:", data.shape)
data.info()

## 3. Data Preprocessing

In this step, missing values, duplicate values, and categorical variables are handled. Categorical values are converted into numerical form using Label Encoding.


In [ ]:
# ===============================
# 3. Data Preprocessing
# ===============================

print("Missing values before preprocessing:")
print(data.isnull().sum())

print("
Duplicate rows:", data.duplicated().sum())

df = data.copy()

# Encode categorical columns
categorical_cols = ['Gender', 'Married', 'Education', 'Employment_Status', 'Property_Area']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("
Data after encoding:")
df.head()

## 4. Exploratory Data Analysis (EDA)

EDA helps us understand the relationship between applicant details and loan approval status.


In [ ]:
# Loan Status Distribution
plt.figure(figsize=(6,4))
data['Loan_Status'].value_counts().plot(kind='bar')
plt.title('Loan Status Distribution')
plt.xlabel('Loan Status (0 = Rejected, 1 = Approved)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Credit Score vs Loan Status
plt.figure(figsize=(6,4))
data.groupby('Loan_Status')['Credit_Score'].mean().plot(kind='bar')
plt.title('Average Credit Score by Loan Status')
plt.xlabel('Loan Status')
plt.ylabel('Average Credit Score')
plt.show()

In [ ]:
# Income vs Loan Status
plt.figure(figsize=(6,4))
data.groupby('Loan_Status')['Applicant_Income'].mean().plot(kind='bar')
plt.title('Average Income by Loan Status')
plt.xlabel('Loan Status')
plt.ylabel('Average Applicant Income')
plt.show()

## 5. Feature Selection and Scaling

The target variable is `Loan_Status`. The remaining columns are used as input features.


In [ ]:
# ===============================
# 5. Feature Selection
# ===============================

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

## 6. Model Training

Three classification models are trained:

- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier


In [ ]:
# ===============================
# 6. Train Models
# ===============================

models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")

## 7. Model Evaluation

The best model is selected based on accuracy and classification performance.


In [ ]:
# Compare model accuracy
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': list(results.values())
}).sort_values(by='Accuracy', ascending=False)

results_df

In [ ]:
# Visualize model comparison
plt.figure(figsize=(7,4))
plt.bar(results_df['Model'], results_df['Accuracy'])
plt.title('Model Accuracy Comparison')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Best model evaluation
best_model_name = results_df.iloc[0]['Model']
print("Best Model:", best_model_name)

best_model = models[best_model_name]

if best_model_name == 'Logistic Regression':
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred_best))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

## 8. Loan Eligibility Prediction Function

This function takes user details and predicts whether the loan should be approved or rejected.


In [ ]:
# ===============================
# 8. Prediction Function
# ===============================

def predict_loan_eligibility(user_data):
    input_df = pd.DataFrame([user_data])
    input_df['Debt_to_Income_Ratio'] = input_df['Existing_EMI'] / input_df['Applicant_Income']
    
    for col in categorical_cols:
        if col in input_df.columns:
            input_df[col] = label_encoders[col].transform(input_df[col])
    
    input_df = input_df[X.columns]
    
    if best_model_name == 'Logistic Regression':
        input_scaled = scaler.transform(input_df)
        prediction = best_model.predict(input_scaled)[0]
        probability = best_model.predict_proba(input_scaled)[0][1]
    else:
        prediction = best_model.predict(input_df)[0]
        probability = best_model.predict_proba(input_df)[0][1]
    
    if prediction == 1:
        status = 'Loan Approved'
    else:
        status = 'Loan Rejected'
    
    return status, round(probability, 2)

sample_user = {
    'Age': 30,
    'Gender': 'Female',
    'Married': 'No',
    'Education': 'Graduate',
    'Employment_Status': 'Salaried',
    'Applicant_Income': 65000,
    'Loan_Amount': 400000,
    'Loan_Term_Months': 240,
    'Credit_Score': 750,
    'Existing_EMI': 10000,
    'Property_Area': 'Urban'
}

status, probability = predict_loan_eligibility(sample_user)
print("Prediction:", status)
print("Approval Probability:", probability)

## 9. AI Loan Advisory Chatbot

A simple rule-based chatbot is created to answer common loan-related queries and provide advisory responses.


In [ ]:
# ===============================
# 9. Simple NLP-Based Chatbot
# ===============================

def loan_chatbot(user_query):
    query = user_query.lower()
    
    if 'eligibility' in query or 'eligible' in query:
        return 'Loan eligibility mainly depends on income, credit score, existing EMI, employment status, and repayment capacity.'
    
    elif 'credit score' in query:
        return 'A good credit score is usually above 700. Higher credit score increases the chances of loan approval.'
    
    elif 'documents' in query:
        return 'Common loan documents include identity proof, address proof, income proof, bank statement, and employment details.'
    
    elif 'interest' in query:
        return 'Loan interest rate depends on bank policy, credit score, loan amount, income, and repayment history.'
    
    elif 'emi' in query:
        return 'EMI depends on loan amount, interest rate, and loan tenure. Lower existing EMI improves loan approval chances.'
    
    elif 'reject' in query or 'rejected' in query:
        return 'Loan can be rejected due to low credit score, low income, high existing EMI, unstable employment, or incorrect documents.'
    
    elif 'improve' in query:
        return 'To improve loan approval chances, maintain a good credit score, reduce existing EMI, increase income stability, and provide correct documents.'
    
    else:
        return 'I can help you with loan eligibility, credit score, EMI, documents, interest rate, and loan approval guidance.'

# Test chatbot
queries = [
    'What is loan eligibility?',
    'What credit score is good for loan?',
    'Which documents are required?',
    'Why loan gets rejected?',
    'How can I improve approval chances?'
]

for q in queries:
    print('User:', q)
    print('Chatbot:', loan_chatbot(q))
    print('-' * 60)

## 10. Final Advisory Function

This combines ML prediction and chatbot-style recommendation.


In [ ]:
# ===============================
# 10. Final Advisory System
# ===============================

def loan_advisory_system(user_data):
    status, probability = predict_loan_eligibility(user_data)
    
    print("Loan Advisory Result")
    print("--------------------")
    print("Prediction:", status)
    print("Approval Probability:", probability)
    
    if status == 'Loan Approved':
        print("Advice: Your loan approval chances are good. Maintain your credit score and submit correct documents.")
    else:
        print("Advice: Your loan approval chances are low. Try improving credit score, reducing existing EMI, or applying for a lower loan amount.")

loan_advisory_system(sample_user)

## Final Summary

The AI Loan Advisory Chatbot project successfully predicts loan eligibility using machine learning classification models and provides basic loan-related guidance through a chatbot interface. The system uses applicant details such as income, credit score, loan amount, employment status, and EMI to predict approval chances.

## Key Outcomes

- Built a loan eligibility prediction model
- Applied preprocessing and feature engineering
- Compared multiple ML models
- Created a simple NLP-based chatbot
- Generated personalized loan advice

## Conclusion

This project can help banks, financial institutions, and customers by providing quick loan eligibility checks and basic advisory support. In the future, it can be improved by adding real-time bank APIs, advanced NLP models, multilingual support, and secure user authentication.
